<a href="https://colab.research.google.com/github/headhuncho1234/HW/blob/main/CIS8694_Fall25_Assignment3Mmotuba_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let’s imagine we have a map with several cities connected by roads. Each road has a specific distance (or cost) associated with it. Our goal is to train an agent to find the shortest path from a starting city to a destination city. The map below shows the source(Nairobi), destination(Kampala), and the various towns surrounding these capitals. We want to implement Q-Learning for Pathfinding.

In [ ]:
!pip install googlemaps

  Preparing metadata (setup.py) ... done


**Question 1: The first step is to define the RL environment as shown below. However, the Reward component is still missing. Please READ THE PRORAM and help explain what is the reward. Note that the overall goal of the agent is to find the shortest path.**

*   States: The cities on the map, eg Nairobi, Kampala, Nakuru, Jinja, etc
*   Actions: Moving from one city to another via a connecting road. For example, from Nairobi, you can choose to go to Thika, Naivasha, or Machakos.
*   Rewards: ?




In [ ]:
print (len(("TYPE YOUR EXPLANATION HERE").split()))

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import googlemaps
import random

# 35 cities between Nairobi and Kampala
CITIES = [
     "Nairobi", "Kitale", "Kisumu", "Machakos", "Thika", "Nakuru", "Bungoma",
    "Naivasha", "Kakamega", "Webuye", "Iten", "Eldoret", "Busia", "Kericho",
    "Thika", "Narok", "Malaba", "Narok", "Kericho", "Kampala", "Jinja", "Busia",
    "Mbale", "Tororo", "Lwakhakha", "Suam", "Entebbe", "Siaya", "Eldama Ravine",
    "Bomet", "Bondo", "Ahero", "Muhoroni", "Butere", "Ugunja", "Chwele"
]



def prepare_cities_data(g_maps_client) -> gpd.GeoDataFrame:
    cities_locations = []
    for city in CITIES:
        geocode_result = g_maps_client.geocode(f"{city}, USA")
        location = {
            "Label": city,
            "colors": random.choice(
                ["black", "red", "green", "blue", "purple", "orange"]
            ),
        }
        try:
            if not geocode_result:
                print(f"Geocoding returned no results for {city}")
                continue
            loc = geocode_result[0]["geometry"]["location"]  # {'lat': ..., 'lng': ...}
            location.update(loc)
            cities_locations.append(location)
        except Exception as e:
            print(f"Failed to get coordinates for {city}: {e}")
    cities_locations_df = pd.DataFrame(cities_locations)
    cities_locations_gdf = gpd.GeoDataFrame(
        cities_locations_df,
        geometry=gpd.points_from_xy(
            cities_locations_df["lng"], cities_locations_df["lat"]
        ),
        crs="EPSG:4326",
    )
    return cities_locations_gdf

With coordinates, we can now get the distances between all the city pairs again using Google Maps.

In [ ]:
def get_origin_destination_cost_matrix(
    cities_locations_gdf: gpd.GeoDataFrame, g_maps_client
) -> np.ndarray:
    """
    Build an NxN driving-distance matrix (in meters) between all city pairs.
    """
    n = len(cities_locations_gdf)
    distances = np.zeros((n, n))

    # Prebuild "lat,lng" strings
    cities_locations_gdf["coord"] = (
        cities_locations_gdf["lat"].astype(str)
        + ","
        + cities_locations_gdf["lng"].astype(str)
    )

    for i in range(n):
        for j in range(n):
            # Skip same city to avoid unnecessary API calls
            if i == j:
                continue

            origin = cities_locations_gdf["coord"].iloc[i]
            destination = cities_locations_gdf["coord"].iloc[j]

            try:
                maps_api_result = g_maps_client.directions(
                    origin,
                    destination,
                    mode="driving",
                )

                if not maps_api_result:
                    print(f"No directions result for {origin} -> {destination}")
                    continue

                leg = maps_api_result[0]["legs"][0]
                distances[i, j] = leg["distance"]["value"]  # in meters

            except Exception as e:
                print(f"Failed to get distance for {origin} -> {destination}: {e}")

    distances = distances.astype(int)

    # Set distance between same town to infinite for optimization
    distances = np.where(distances == 0, float("inf"), distances)
    return distances

**Question 2: When implementing a Q-learning algorithm, we need to define several hyperparameters.
Please explain what a hyperparameter is, and describe the key hyperparameters required for this Q-learning algorithm.
Then, include Python code that assigns your preferred values to each hyperparameter. Note: please use the variable names under question 3.**


In [ ]:

#insert your code here

**Question 3: Next step is to initialize the Q table and update it. Please insert your code to complete the program for this step.**

In [ ]:

EPSILON = 0.2
def get_q_learning_cost_table(
    cities_locations_gdf: gpd.GeoDataFrame,
    num_episodes: int,
    start_city_index: str,
    end_city_index: str,
    distances: np.ndarray,
) -> np.ndarray:
    q_table = #insert your code here and initialize all values as 0
    for _ in range(num_episodes):
        current_city = start_city_index
        while current_city != end_city_index:
            action = select_next_action(distances, current_city, q_table)
            if action is None:
                break
            next_city = action
            update_q_table(q_table, distances, current_city, action, next_city)

            current_city = next_city
            if current_city == end_city_index:
                break
    return q_table

def select_next_action(
    distances: np.ndarray, current_city: int, q_table: np.ndarray
) -> int:
    possible_actions = (
        np.where(distances[current_city, :] > 0)[0]
        if np.random.uniform(0, 1) < EPSILON
        else np.where(
            q_table[current_city, :] == np.max(q_table[current_city, :])
        )[0]
    )
    if len(possible_actions) == 0:
        return
    return np.random.choice(possible_actions)

def update_q_table(
    q_table: np.ndarray,
    distances: np.ndarray,
    current_city: int,
    action: int,
    next_city: int,
) -> None:
    # the reward is negative since the goal is to have minimum distance
    reward = -distances[current_city, next_city]
    current_state_action_value = q_table[current_city, action]
    next_state_action_value = np.max(q_table[next_city, :])

    q_table[current_city, action] = (
        1 - LEARNING_RATE
    ) * current_state_action_value + LEARNING_RATE * (
        reward + DISCOUNT_FACTOR * next_state_action_value
    )


**Question 4: Explain how to determine the shortest path by interpreting the learned Q-values, and describe the steps required to extract the optimal route from the Q-table. Hint: Please read program below.**


In [ ]:
 print (len(("TYPE YOUR EXPLANATION HERE").split()))

In [ ]:
def get_shortest_path(
    q_table: np.ndarray, start_city_index: int, end_city_index: int
) -> list:
    shortest_path = [start_city_index]
    current_city = start_city_index
    while current_city != end_city_index:
        next_city = np.argmax(q_table[current_city, :])
        shortest_path.append(next_city)
        current_city = next_city
    route = [(start, dest) for start, dest in zip(shortest_path, shortest_path[1:])]
    return shortest_path, route


With the logic to get the Q-table and extract the shortest path between source and destination ready, we can now use them to get the optimal path between two cities given the distances between a set of cities.

In [ ]:
def get_optimal_path(
    cities_locations_gdf: gpd.GeoDataFrame,
    distances: np.ndarray,
    start_city: str,
    end_city: str,
) -> list:
    # Find indices for start and end cities
    try:
        start_city_index = cities_locations_gdf[
            cities_locations_gdf["Label"] == start_city
        ].index[0]
    except IndexError:
        raise ValueError(f"Start city '{start_city}' not found in GeoDataFrame.")

    try:
        end_city_index = cities_locations_gdf[
            cities_locations_gdf["Label"] == end_city
        ].index[0]
    except IndexError:
        raise ValueError(f"End city '{end_city}' not found in GeoDataFrame.")

    # These functions must exist elsewhere in your code
    q_table = get_q_learning_cost_table(
        cities_locations_gdf, 300, start_city_index, end_city_index, distances
    )

    q_table_df = pd.DataFrame(
        data=q_table,
        index=cities_locations_gdf["Label"],
        columns=cities_locations_gdf["Label"],
    )

    shortest_path_indices, route = get_shortest_path(
        q_table, start_city_index, end_city_index
    )

    shortest_path_labels = [
        cities_locations_gdf["Label"][city_index] for city_index in shortest_path_indices
    ]
    shortest_path_str = "->".join(shortest_path_labels)
    return shortest_path_str, route


SyntaxError: incomplete input (ipython-input-433369296.py, line 39)

**Question 5: Run your program and review the results.
Verify whether the computed optimal path is correct.**

In [ ]:
print (len(("TYPE YOUR EXPLANATION HERE").split()))

In [ ]:
def get_distance(distances: np.array, route: list) -> int:
    route_distance = 0
    for origin, destination in route:
        route_distance += distances[origin][destination]
    return int(route_distance)


# ------------------ USAGE EXAMPLE ------------------

# Replace with your real API key in your actual code,
# but don't commit it to GitHub or share publicly.
API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"

g_maps_client=googlemaps.Client(key=API_KEY)


cities_locations_gdf = prepare_cities_data(g_maps_client)
distances = get_origin_destination_cost_matrix(cities_locations_gdf, g_maps_client)

# Getting the optimal path between Nairobi and Nairobi

shortest_path, route = get_optimal_path(cities_locations_gdf, distances, "Nairobi", "Kampala")


print(shortest_path)
print(get_distance(distances, route))

**Question 5: Could you describe the advantages and limitations of using Q-learning for this shortest-path problem?**

In [ ]:
print (len(("TYPE YOUR EXPLANATION HERE").split()))

**Submission:**

For google colab users,
Download your final notebook as .ipynb.
Download your final notebook as .pdf.
Submit both files to iCollege site with name **CIS8694-Fall25-Assignment3-Part 2-yourFirstInitialLastName( .ipynb / .pdf)**
Share a google drive link to your .ipynb in the comment of submission.



For anaconda users,
Download your final notebook as HTML
Download your final notebook as .ipynb.
Submit both files to iCollege site with name **CIS8694-Fall25-Assignment3-Part 2-yourFirstInitialLastName( .ipynb / .html)**
Share a google drive link to your .ipynb in the comment of submission.

